In [ ]:
%pip install numpy matplotlib qiskit qiskit-aer scipy --quiet

In [ ]:
from __future__ import annotations
import math
from dataclasses import dataclass, field
from typing import Callable, Sequence
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector

In [ ]:
#Erasure Conversion

def encoder() -> QuantumCircuit:
    logical = QuantumRegister(1, name="in")
    data = QuantumRegister(3, name="anc")
    qc = QuantumCircuit(logical, data, name="Encoder")
    qc.h(logical[0])
    qc.cx(logical[0], data[0])
    qc.cx(logical[0], data[1])
    qc.cx(logical[0], data[2])
    qc.cx(data[0], data[2])
    return qc

In [ ]:
#Decoder

@dataclass
class EraSyn:
    erased: set[int]
    stabilizers: list[set[int]]
    syn_bits: list[int]

def decoder(syn:EraSyn) -> dict[int, int]:
    erased = set(syn.erased)
    stab = [set(s) for s in syn.stabilizers]
    syn = list(syn.syn_bits)
    correction: dict[int, int] = {q: 0 for q in erased}

    changed = True
    while changed and erased:
        changed = False
        for i, (s, b) in enumerate(zip(stab, syn)):
            intersect = s & erased
            if len(intersect) == 1:
                q = next(iter(intersect))
                correction[q] ^= b
                for j, s2 in enumerate(stab):   #Pass forward the correction to the remaining stabilizers
                    if q in s2:
                        syn[j] ^= correction[q]
                        s2.discard(q)
                erased.discard(q)
                changed = True
                break

    return correction

In [ ]:
#Repeat until success (RUS)

@dataclass
class RUSResult:
    success: bool
    attempts: int

def RUS(attempt:Callable[[], bool], max_attempts:int = 50, rng:np.random.Generator | None = None) -> RUSResult:
    for i in range(1, max_attempts+1):
        if attempt():
            return RUSResult(success=True, attempts=i)
    return RUSResult(success=False, attempts=max_attempts)

def heralded_retry(quantcirc:QuantumCircuit, measurement:tuple[int, int], max_attempts:int = 8) -> QuantumCircuit:
    cntr = max(1, int(math.ceil(math.log2(max_attempts+1))))
    herald = ClassicalRegister(1, name="herald")
    counter = ClassicalRegister(cntr, name="attempt")

    exec_qc = QuantumCircuit(*quantcirc.qregs, herald, counter, name="HeraldedRetryExecutor")  #Copying all quantum registers
    herald_qubit_idx, _herald_cbit_idx = measurement

    refresh = quantcirc.inverse()
    with exec_qc.while_loop((herald, 0)):   #Pre-initialize the herald
        exec_qc.compose(quantcirc, inplace=True)
        exec_qc.measure(exec_qc.qubits[herald_qubit_idx], herald[0])
        with exec_qc.if_test((herald, 0)):  #Refresh the circuit
            exec_qc.compose(refresh, inplace=True)

    return exec_qc

In [ ]:
#Surface code Correction

def surface_code(dist:int) -> QuantumCircuit:
    n_data = dist*dist
    z_anc = (dist-1)*(dist-1)
    x_anc = (dist-1)*(dist-1)

    data = QuantumRegister(n_data, name="data")
    z = QuantumRegister(z_anc, name="z_ancilla")
    x = QuantumRegister(x_anc, name="x_ancilla")
    z_class = ClassicalRegister(z_anc, name="z_syndrome")
    x_class = ClassicalRegister(x_anc, name="x_syndrome")
    qc = QuantumCircuit(data, z, x, z_class, x_class, name=f"SurfaceCode-d={dist}")

    for i in range(z_anc):  #For a Z-plaquette, each z ancilla couples to its 4 neighbors via CX
        row, col = i//(dist-1), i%(dist-1)
        neighbors = [
            row*dist + col,
            row*dista + col+1,
            (row+1)*dist + col,
            (row+1)*dist + col+1]

        for nb in neighbors:
            qc.cx(data[nb], z[i])
        qc.measure(z[i], z_class[i])

    qc.barrier()

    for i in range(x_anc):  #Similarly for a X-plaquette, it is via H-sandwiched between CX via ancilla
        row, col = i//(dist-1), i%(dist-1)
        neighbors = [
            row*dist + col,
            row*dista + col+1,
            (row+1)*dist + col,
            (row+1)*dist + col+1]

        qc.h(x_anc[i])
        for nb in neighbors:
            qc.cx(x[i], data[nb])
        qc.h(x[i])
        qc.measure(x[i], x_class[i])

    return qc

def threshold_check(erasure:float) -> tuple[float, bool]:
    threshold = 0.5 #demo value
    return threshold, erasure < threshold

In [ ]:
#Dual-rail erasure check

def biased_threshold(erasure:float, pauli_thrshld:float = 0.00937, erasure_thrshld:float = 0.0415) -> float:
    assert 0.0 <= erasure <= 1.0
    return (erasure*erasure_thrshld + (1.0 - erasure)*pauli_thrshld)

In [ ]:
#Routing Flag qubit parity check

def flag_parity(n_data:int) -> QuantumCircuit:
    data = QuantumRegister(n_data, name="data")
    syn = QuantumRegister(1, name="syndrome")
    flag = QuantumRegister(1, name="flag")
    class_syn = ClassicalRegister(1, name="syndrome_bit")
    class_flag = ClassicalRegister(1, name="flag_bit")
    qc = QuantumCircuit(data, syn, flag, class_syn, class_flag, name="ParityCheck")

    qc.h(syn[0])
    qc.cx(syn[0], data[0])
    qc.cx(syn[0], flag[0])  #Flag

    for d in range(1, n_data-1):
        qc.cx(syn[0], data[d])

    qc.cx(syn[0], flag[0])  #Unflag
    qc.cx(syn[0], data[n_data-1])
    qc.h(syn[0])
    qc.measure(syn[0], class_syn[0])
    qc.measure(flag[0], class_flag[0])

    return qc

In [ ]:
#Small demo

def _demo_layer4():
    print("Layer 4")

    enc = encoder()
    qc = QuantumCircuit(4)
    qc.compose(enc, qubits=[0, 1, 2, 3], inplace=True)
    sv_enc = Statevector.from_instruction(qc)
    print(f"4-qubit state with norm{np.linalg.norm(sv_enc.data):.6f}")
    print("Encoded state:")
    for i, a in enumerate(sv_enc.data):
        if abs(a) > 1e-8:
            print(f"|{i:04b}> amplitude {a.real:+.4f}{a.imag:+.4f}j")

    synd = EraSyn(erased={0, 2}, stabilizers=[{0, 1}, {1, 2}, {2, 3}], syn_bits=[1, 0, 1])
    corr = decoder(synd)
    print(f"Decoder on 4-qubit repetition code with erasure: {sorted(synd.erased)}")
    print(f"Corrections: {corr}")

    for era in (0.1, 0.3, 0.49, 0.5, 0.51, 0.7):
        thr, ok = threshold_check(era)
        print(f"Erasure rate {era:.2f}: {'BELOW' if ok else 'ABOVE'} threshold")

    print("Effective threshold with erasure conversion:")
    for era in (0.0, 0.25, 0.5, 0.75, 0.98, 1.0):
        thr = biased_threshold(era)
        print(f"Erasure fraction {era:.2f}: threshold = {thr:.4f}")

    rng = np.random.default_rng(0)
    result = RUS(lambda: rng.random() < 0.2, rng=rng)
    print(f"RUS: success={result.success}, " f"attempts={result.attempts}")

    qr = QuantumRegister(1)
    quant_circ = QuantumCircuit(qr, name="DemoCircuit")
    quant_circ.h(qr[0])
    hrld = heralded_retry(quant_circ, measurement=(0, 0), max_attempts=8)

    def count_itr(circ, name):
        return sum(1 for instr in circ.data if instr.operation.name == name)

    print(f"Heralded retry circuit:")
    print(f"Qubits = {hrld.num_qubits}")
    print(f"Classical bits = {hrld.num_clbits}")
    print(f"Iterations = {count_itr(hrld)}")

    print("Surface code stabilizer routine:")
    for d in (3, 5, 7):
        qc_sc = surface_code(dist=d)
        print(f"Dist = {d}: qubits = {qc_sc.num_qubits:>3}, " f"depth = {qc_sc.depth():>4}")

    print("Flag qubit parity check routine:")
    for w in (2, 3, 4, 6):
        qc_f = flag_parity(w)
        print(f"Weight = {w}: depth = {qc_f.depth()}")

In [ ]:
if __name__ == "__main__":
    _demo_layer4()